# Bayesian A/B Testing

This notebook demonstrates Bayesian inference for A/B tests using the Beta-Binomial conjugate model.

**Why Bayesian?**
- Direct probability statements: "There is a 95% probability B is better than A"
- No fixed sample sizes needed; you can peek at results
- Decision rules based on expected loss rather than p-values
- Natural incorporation of prior knowledge

**Model:** For conversion data, we use a Beta prior on the conversion rate. As data arrives, the posterior is also Beta (conjugacy), making computation exact and fast.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from src.bayesian import BayesianABTest

np.random.seed(42)
bayes = BayesianABTest(n_simulations=200000)
print("BayesianABTest loaded successfully.")

## Simulating Experiment Data

We simulate a scenario where:
- Control has a 5.0% conversion rate
- Treatment has a 5.8% conversion rate (16% relative lift)
- Each group has 3,000 users

In [ ]:
# Simulate experiment data
true_rate_control = 0.050
true_rate_treatment = 0.058
n_users = 3000

control_conversions = np.random.binomial(n_users, true_rate_control)
treatment_conversions = np.random.binomial(n_users, true_rate_treatment)

print(f"Simulated experiment results:")
print(f"  Control:   {control_conversions}/{n_users} = {control_conversions/n_users:.4f}")
print(f"  Treatment: {treatment_conversions}/{n_users} = {treatment_conversions/n_users:.4f}")
print(f"  Observed lift: {(treatment_conversions - control_conversions) / control_conversions * 100:.1f}%")

## Beta-Binomial Analysis

We use an uninformative Beta(1,1) prior (uniform on [0,1]) and compute the posterior for each variant. The `BayesianABTest` class uses Monte Carlo sampling to estimate:
- P(B > A): probability treatment is better
- Expected loss: the average regret of choosing the wrong variant

In [ ]:
result = bayes.beta_binomial(
    control_conversions=control_conversions,
    control_total=n_users,
    treatment_conversions=treatment_conversions,
    treatment_total=n_users,
    prior_alpha=1.0,
    prior_beta=1.0,
    loss_threshold=0.001
)

print("Bayesian Analysis Results:")
print(f"  P(Treatment > Control):  {result.prob_b_better:.4f}")
print(f"  Posterior mean (Control): {result.posterior_mean_a:.4f}")
print(f"  Posterior mean (Treatment): {result.posterior_mean_b:.4f}")
print(f"  95% Credible Interval for diff: [{result.credible_interval[0]:.4f}, {result.credible_interval[1]:.4f}]")
print(f"  Expected loss (choose A): {result.expected_loss_a:.6f}")
print(f"  Expected loss (choose B): {result.expected_loss_b:.6f}")
print(f"  Risk threshold met: {result.risk_threshold_met}")

## Posterior Visualization

We plot the posterior Beta distributions for both variants. The separation between the two distributions indicates the strength of evidence.

In [ ]:
# Compute posterior parameters
alpha_a = 1 + control_conversions
beta_a = 1 + (n_users - control_conversions)
alpha_b = 1 + treatment_conversions
beta_b = 1 + (n_users - treatment_conversions)

x = np.linspace(0.03, 0.08, 1000)
pdf_a = stats.beta.pdf(x, alpha_a, beta_a)
pdf_b = stats.beta.pdf(x, alpha_b, beta_b)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Posterior distributions
ax = axes[0]
ax.plot(x, pdf_a, 'b-', linewidth=2, label=f'Control (posterior mean={alpha_a/(alpha_a+beta_a):.4f})')
ax.fill_between(x, pdf_a, alpha=0.2, color='blue')
ax.plot(x, pdf_b, 'r-', linewidth=2, label=f'Treatment (posterior mean={alpha_b/(alpha_b+beta_b):.4f})')
ax.fill_between(x, pdf_b, alpha=0.2, color='red')
ax.set_xlabel('Conversion Rate', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Posterior Distributions of Conversion Rate', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Right: Distribution of the difference (B - A)
ax = axes[1]
samples_a = np.random.beta(alpha_a, beta_a, 100000)
samples_b = np.random.beta(alpha_b, beta_b, 100000)
diff = samples_b - samples_a
ax.hist(diff, bins=80, density=True, alpha=0.7, color='green', edgecolor='none')
ax.axvline(x=0, color='black', linestyle='--', linewidth=1.5, label='Zero effect')
ax.axvline(x=np.mean(diff), color='darkgreen', linestyle='-', linewidth=2, label=f'Mean diff = {np.mean(diff):.4f}')
ax.set_xlabel('Difference (Treatment - Control)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Posterior Distribution of Effect Size', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Decision-Making with Expected Loss

Instead of relying solely on P(B > A), we use **expected loss** for decisions. The expected loss of choosing B is the average amount we lose if B is actually worse. A common threshold is 0.1% (0.001 in absolute terms).

We simulate the decision process over time as data accumulates:

In [ ]:
# Track metrics over time as data accumulates
checkpoints = [100, 250, 500, 1000, 1500, 2000, 2500, 3000]
loss_threshold = 0.001

prob_b_history = []
loss_b_history = []

# Simulate cumulative arrivals
all_control = np.random.binomial(1, true_rate_control, n_users)
all_treatment = np.random.binomial(1, true_rate_treatment, n_users)

print(f"{'N/group':>8} {'Conv_A':>8} {'Conv_B':>8} {'P(B>A)':>8} {'Loss(B)':>10} {'Decision':>12}")
print("-" * 62)

for n in checkpoints:
    conv_a = all_control[:n].sum()
    conv_b = all_treatment[:n].sum()
    
    r = bayes.beta_binomial(
        control_conversions=int(conv_a),
        control_total=n,
        treatment_conversions=int(conv_b),
        treatment_total=n,
        loss_threshold=loss_threshold
    )
    
    prob_b_history.append(r.prob_b_better)
    loss_b_history.append(r.expected_loss_b)
    
    if r.expected_loss_b < loss_threshold:
        decision = "Ship B"
    elif r.expected_loss_a < loss_threshold:
        decision = "Keep A"
    else:
        decision = "Continue"
    
    print(f"{n:>8} {conv_a:>8} {conv_b:>8} {r.prob_b_better:>8.4f} {r.expected_loss_b:>10.6f} {decision:>12}")

In [ ]:
# Plot decision metrics over time
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(checkpoints, prob_b_history, 'bo-', linewidth=2, markersize=6)
ax1.axhline(y=0.95, color='green', linestyle='--', alpha=0.7, label='95% threshold')
ax1.axhline(y=0.5, color='gray', linestyle=':', alpha=0.5)
ax1.set_xlabel('Sample Size per Group', fontsize=12)
ax1.set_ylabel('P(Treatment > Control)', fontsize=12)
ax1.set_title('Probability of Treatment Being Better', fontsize=13)
ax1.set_ylim(0, 1.05)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

ax2.plot(checkpoints, loss_b_history, 'rs-', linewidth=2, markersize=6)
ax2.axhline(y=loss_threshold, color='green', linestyle='--', alpha=0.7, label=f'Threshold = {loss_threshold}')
ax2.set_xlabel('Sample Size per Group', fontsize=12)
ax2.set_ylabel('Expected Loss (choosing B)', fontsize=12)
ax2.set_title('Expected Loss Over Time', fontsize=13)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal decision: Expected loss of choosing B = {loss_b_history[-1]:.6f}")
if loss_b_history[-1] < loss_threshold:
    print(f"Below threshold ({loss_threshold}) -> Safe to ship Treatment.")
else:
    print(f"Above threshold ({loss_threshold}) -> Need more data or Treatment is not clearly better.")

## Lift Distribution

Finally, we compute the full posterior distribution of relative lift to answer questions like: "What is the probability that treatment improves conversions by more than 10%?"

In [ ]:
lift = bayes.compute_lift_distribution(
    control_conversions=control_conversions,
    control_total=n_users,
    treatment_conversions=treatment_conversions,
    treatment_total=n_users
)

print("Lift Distribution Summary:")
print(f"  Mean relative lift:    {lift['mean_lift']*100:.2f}%")
print(f"  Median relative lift:  {lift['median_lift']*100:.2f}%")
print(f"  95% CI:                [{lift['ci_95'][0]*100:.2f}%, {lift['ci_95'][1]*100:.2f}%]")
print(f"  P(lift > 0%):          {lift['prob_positive_lift']:.4f}")
print(f"  P(lift > 5%):          {lift['prob_lift_gt_5pct']:.4f}")
print(f"  P(lift > 10%):         {lift['prob_lift_gt_10pct']:.4f}")